In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

In [3]:
ID_COL = 'product_id'
DROP_COL = 'product_name' # Not required for model training
numeric_cols = ['product_weight', 'fragility_index']
categorical_cols = ['category', 'shipping_type']
all_process_cols = numeric_cols + categorical_cols

In [8]:
dfp = pd.read_csv("packaging_products_sample.csv")
df_process = dfp[[ID_COL, DROP_COL] + all_process_cols].copy()

In [9]:
dfp.head()

,product_id,product_name,category,product_weight,fragility_index,shipping_type
0,PRD_20001,Water Bottle SKU-0,Sports,0.726424,4,Air
1,PRD_20002,Area Rug (Small) SKU-1,Home Goods,1.425100,5,Sea
2,PRD_20003,Yoga Mat SKU-2,Sports,0.713777,2,Sea
3,PRD_20004,Hair Spray Can SKU-3,Cosmetics,0.387991,3,Air
4,PRD_20005,Backpack SKU-4,Apparel,0.466723,2,Sea


In [10]:
missing_info = df_process[all_process_cols].isnull().sum()
missing_data_rows = ""
for col, count in missing_info.items():
    if count > 0 and col in numeric_cols:
        # Based on previous EDA, these have missing values
        missing_data_rows += f"| {col} | {int(count)} | Median Imputation (Numeric) |\n"
    elif count > 0 and col in categorical_cols:
        # Although not expected to be missing, include strategy
        missing_data_rows += f"| {col} | {int(count)} | Mode Imputation (Categorical) |\n"

In [11]:
missing_report = f"""
# Products Dataset - Missing Value Report
---
## Summary of Missing Values

| Column | Missing Count | Strategy |
| :--- | :--- | :--- |
{missing_data_rows if missing_data_rows else '| N/A | 0 | N/A |'}

## Decisions
1. **Numeric features** (Product Weight, Fragility Index): Impute using the **Median** strategy to ensure robust handling of missing values.
2. **Categorical features** (Category, Shipping Type): Impute using the **Mode (most frequent)** strategy.
3. **No rows were removed.**
"""
# Save Missing Value Report (Deliverable)
with open('products_missing_value_report.md', 'w') as f:
    f.write(missing_report)
print(f"Generated missing value report: {'products_missing_value_report.md'}")

Generated missing value report: products_missing_value_report.md


In [12]:
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

In [13]:
df_process[numeric_cols] = num_imputer.fit_transform(df_process[numeric_cols])

In [14]:
df_process[categorical_cols] = cat_imputer.fit_transform(df_process[categorical_cols])

In [15]:
df_process.isnull().sum()

product_id         0
product_name       0
product_weight     0
fragility_index    0
category           0
shipping_type      0
dtype: int64

In [17]:
df_process.to_csv('cleaned_products_dataset.csv', index=False)
print(f"Exported cleaned dataset: {'cleaned_products_dataset.csv'}")

Exported cleaned dataset: cleaned_products_dataset.csv


In [18]:
df_cleaned = pd.read_csv('cleaned_products_dataset.csv')

In [19]:
df_cleaned.head()

,product_id,product_name,product_weight,fragility_index,category,shipping_type
0,PRD_20001,Water Bottle SKU-0,0.726424,4.0,Sports,Air
1,PRD_20002,Area Rug (Small) SKU-1,1.425100,5.0,Home Goods,Sea
2,PRD_20003,Yoga Mat SKU-2,0.713777,2.0,Sports,Sea
3,PRD_20004,Hair Spray Can SKU-3,0.387991,3.0,Cosmetics,Air
4,PRD_20005,Backpack SKU-4,0.466723,2.0,Apparel,Sea


In [21]:
import joblib
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from io import StringIO

In [22]:
ohe_cols = categorical_cols
ohe_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_data = ohe_encoder.fit_transform(df_cleaned[ohe_cols])

In [23]:
ohe_feature_names = ohe_encoder.get_feature_names_out(ohe_cols)
df_ohe = pd.DataFrame(ohe_data, columns=ohe_feature_names)

In [24]:
df_encoded = pd.concat([df_cleaned.drop(columns=ohe_cols + [DROP_COL]), df_ohe], axis=1)

In [26]:
df_encoded.head()

,product_id,product_weight,fragility_index,category_Apparel,category_Cosmetics,category_Electronics,category_Food,category_Home Goods,category_Pharmacy,category_Sports,shipping_type_Air,shipping_type_Road,shipping_type_Sea
0,PRD_20001,0.726424,4.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
1,PRD_20002,1.425100,5.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,PRD_20003,0.713777,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,PRD_20004,0.387991,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,PRD_20005,0.466723,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [27]:
scaling_cols = numeric_cols
scaler = MinMaxScaler()
df_encoded[scaling_cols] = scaler.fit_transform(df_encoded[scaling_cols])

In [28]:
joblib.dump(ohe_encoder,'products_ohe_encoder.pkl')
joblib.dump(scaler,'products_scaler.pkl')
print(f"Saved One-Hot Encoder to {'products_ohe_encoder.pkl'}")
print(f"Saved MinMaxScaler to {'products_scaler.pkl'}")

Saved One-Hot Encoder to products_ohe_encoder.pkl
Saved MinMaxScaler to products_scaler.pkl


In [29]:
encoding_report = f"""
# Products Dataset - Encoding and Normalization Report
---
## Encoding Strategy
1. **Nominal Categories** ({', '.join(ohe_cols)}): Used **One-Hot Encoding** (OHE). The fitted encoder object has been saved for reuse during model inference.

## Normalization Strategy
1. **Numeric Features** ({', '.join(scaling_cols)}): Used **MinMaxScaler** to scale features to the range [0, 1]. The fitted scaler object has been saved for reuse during model inference.

## Validation (Final Encoded Dataset)
* **No missing values** remaining.
* **Numeric features** scaled between 0 and 1.
* **Categorical fields** successfully converted to numeric binary features.
"""
with open('products_encoding_normalization_report.md', 'w') as f:
    f.write(encoding_report)
print(f"Generated encoding/normalization report: {'products_encoding_normalization_report.md'}")

Generated encoding/normalization report: products_encoding_normalization_report.md


In [30]:
df_encoded.to_csv('products_final_encoded.csv', index=False)